# 02 — Hybrid Search, RRF, and Reranking

The second half of the funnel: fuse the two retrievers with **Reciprocal Rank
Fusion**, then let a **cross-encoder** re-score the shortlist.

```
retrieve (BM25 + k-NN)  →  RRF fuse  →  trim  →  cross-encoder  →  top-5
```

In [ ]:
import pandas as pd
from opensearch_demo import get_client, wait_for_cluster
from opensearch_demo.search import lexical_search, neural_search, reciprocal_rank_fusion
from opensearch_demo.rerank import rerank
from opensearch_demo import pipeline
from opensearch_demo.config import PipelineParams

INDEX = "products-appliances"
client = get_client(); wait_for_cluster(client)
print(client.count(index=INDEX)["count"], "products")

## Why fuse on *rank* and not score

BM25 scores are unbounded and corpus-dependent; cosine similarity lives in
[-1, 1]. Adding them is meaningless, normalising them is guesswork. RRF
sidesteps both: each list contributes `1 / (k + rank)`.

`_ranks` below shows each document's rank in {0: lexical, 1: neural} —
documents found by **both** lists float to the top even when neither ranked
them first.

In [ ]:
q = "quiet ice machine for home bar"
lex = lexical_search(client, q, k=20, index=INDEX)
vec = neural_search(client, q, k=20, index=INDEX)
fused = reciprocal_rank_fusion([lex, vec], k=60)
pd.DataFrame([{ "title": d["title"][:60], "rrf": round(d["_rrf_score"], 4),
                "lex_rank": d["_ranks"].get(0), "vec_rank": d["_ranks"].get(1)}
              for d in fused[:10]])

In [ ]:
# Weights bias the fusion. 2:1 toward neural — watch identifiers sink and
# paraphrase matches rise. There is no universally right setting; it is a knob
# you tune against judged queries (next session).
fused_w = reciprocal_rank_fusion([lex, vec], k=60, weights=[1.0, 2.0])
pd.DataFrame([{ "title": d["title"][:60],
                "even": next((i+1 for i, x in enumerate(fused) if x["id"] == d["id"]), None),
                "neural-heavy": i + 1}
              for i, d in enumerate(fused_w[:10])])

## The cross-encoder earns its cost on the shortlist

A bi-encoder embeds query and document **separately** — that is what makes
indexing possible. A cross-encoder reads them **jointly** — more accurate,
nothing precomputable. So it only ever sees the trimmed shortlist.
`moved` = retrieval rank − reranked rank.

In [ ]:
shortlist = fused[:10]
rr = rerank(q, shortlist)
pd.DataFrame([{ "title": d["title"][:60], "ce_score": round(d["_ce_score"], 2),
                "was": d["_retrieval_rank"], "moved": f'{d["_rank_delta"]:+d}'}
              for d in rr])

## The whole funnel, traced

In [ ]:
res = pipeline.run(client, "energy efficient wine fridge for 20 bottles",
                   mode="hybrid", index=INDEX, params=PipelineParams())
print(res.explain(), "\n")
pd.DataFrame([{ "title": d["title"][:65], "price": d.get("price"),
                "rating": d.get("average_rating"), "ce": round(d["_ce_score"], 2)}
              for d in res.documents])

## `ef_search`: the one HNSW knob you can turn on a live index

Bigger queue → better recall → slower queries. We measure recall against
**exact** k-NN (a script-score scan — fine for one query on 94k docs, absurd
in production, which is the whole reason HNSW exists).

In [ ]:
import time
from opensearch_demo.embed import encode_query

qv = encode_query("quiet ice machine for home bar").tolist()

exact = client.search(index=INDEX, body={
    "size": 10, "_source": False,
    "query": {"script_score": {"query": {"match_all": {}},
        "script": {"source": "knn_score", "lang": "knn",
                   "params": {"field": "embedding", "query_value": qv,
                              "space_type": "cosinesimil"}}}}})
truth = {h["_id"] for h in exact["hits"]["hits"]}

rows = []
for ef in [10, 25, 50, 100, 200, 400]:
    client.indices.put_settings(index=INDEX, body={"index": {"knn.algo_param.ef_search": ef}})
    t0 = time.perf_counter()
    r = client.search(index=INDEX, body={"size": 10, "_source": False,
        "query": {"knn": {"embedding": {"vector": qv, "k": 10}}}})
    ms = (time.perf_counter() - t0) * 1000
    got = {h["_id"] for h in r["hits"]["hits"]}
    rows.append({"ef_search": ef, "recall@10": len(got & truth) / 10, "latency_ms": round(ms, 1)})
client.indices.put_settings(index=INDEX, body={"index": {"knn.algo_param.ef_search": 100}})
pd.DataFrame(rows).set_index("ef_search")

---
**Where this goes next** (Rashid's next session): judged queries and NDCG —
so “neural-heavy weights look better” becomes a number, not an opinion.
The candidate for that is the Amazon **ESCI** dataset (real queries with graded
relevance labels), joined to this corpus by ASIN.